# Supplementary Figure S4 - Related-Tool Positioning

Tightly scoped capability comparison supporting the manuscript's novelty framing relative to WGCNA, clusterProfiler, ComplexHeatmap, related web tools, and a manual ORA-plus-heatmap workflow. This is positioning evidence, not a performance benchmark: no competing tool is installed, run, timed, or scored on biological output.

## Setup

In [ ]:
# Root-level publication export paths.
from pathlib import Path

PNG_DIR = Path("png") / "supp_fig_4"
PNG_DIR.mkdir(parents=True, exist_ok=True)

print(f"PNG output -> {PNG_DIR}")

## Curated Capability Matrix

Rows are sorted by coverage of the displayed capabilities, using the transparent score **Yes=2**, **Partial=1**, and **No/Not documented=0**. Ties preserve a curated order among related tool types. This keeps HiMaLAYAS at the top because it is the software being positioned and covers every displayed capability, without using unsupervised clustering or implying a performance benchmark.

Values are categorical summaries:

- **Yes**: supported as a documented/package-level capability.
- **Partial**: possible only through custom looping/glue code, or supported only for part of the relevant capability.
- **No**: not part of the tool/workflow's documented role.
- **Not documented**: not clearly established from the sources consulted.

Live web-service availability is omitted from the figure because it is time-sensitive; dated status/provenance remains in the source table generated below.

In [ ]:
import pandas as pd

# Tie-break order is curated; the final plotted order is score-sorted below.
TOOLS = [
    "HiMaLAYAS",
    "Heatmap + manual ORA",
    "clusterProfiler",
    "ComplexHeatmap",
    "WGCNA",
    "VisHiC",
    "funcExplorer",
]

AXES = [
    "Post hoc annotation",
    "Hierarchical zoom",
    "Enrichment test",
    "Multiple-testing control",
    "Matrix-embedded labels",
    "Beyond expression data",
    "Local install / maintained",
]

capability_rows = [
    {
        "tool": "VisHiC",
        AXES[0]: "No",
        AXES[1]: "Partial",
        AXES[2]: "Yes",
        AXES[3]: "Not documented",
        AXES[4]: "Yes",
        AXES[5]: "No",
        AXES[6]: "No",
        "core_positioning": "Historical web tool for enrichment-guided microarray heatmap annotation.",
        "scope_limitation": "Enrichment-guided rather than post hoc on independently defined clusters; expression-focused; no local package located.",
    },
    {
        "tool": "funcExplorer",
        AXES[0]: "No",
        AXES[1]: "Not documented",
        AXES[2]: "Yes",
        AXES[3]: "Partial",
        AXES[4]: "Not documented",
        AXES[5]: "No",
        AXES[6]: "Partial",
        "core_positioning": "Historical web workflow for data-driven functional characterization of expression data.",
        "scope_limitation": "Expression-focused, enrichment-guided workflow; source is recoverable but not a simple local analysis library.",
    },
    {
        "tool": "WGCNA",
        AXES[0]: "Partial",
        AXES[1]: "No",
        AXES[2]: "No",
        AXES[3]: "No",
        AXES[4]: "No",
        AXES[5]: "Partial",
        AXES[6]: "Yes",
        "core_positioning": "Mature package for weighted correlation network/module discovery.",
        "scope_limitation": "Does not perform enrichment testing or matrix-linked term annotation itself; users add separate enrichment/plotting steps.",
    },
    {
        "tool": "clusterProfiler",
        AXES[0]: "Partial",
        AXES[1]: "No",
        AXES[2]: "Yes",
        AXES[3]: "Yes",
        AXES[4]: "No",
        AXES[5]: "Partial",
        AXES[6]: "Yes",
        "core_positioning": "Mature enrichment engine for supplied gene lists or ranked lists.",
        "scope_limitation": "Does not cluster matrices or render significant terms directly on a matrix layout; per-cluster use requires custom looping.",
    },
    {
        "tool": "ComplexHeatmap",
        AXES[0]: "No",
        AXES[1]: "No",
        AXES[2]: "No",
        AXES[3]: "No",
        AXES[4]: "Yes",
        AXES[5]: "Yes",
        AXES[6]: "Yes",
        "core_positioning": "Mature, flexible heatmap and annotation-track renderer.",
        "scope_limitation": "Performs no enrichment testing; statistically significant terms must be computed elsewhere and supplied by the user.",
    },
    {
        "tool": "Heatmap + manual ORA",
        AXES[0]: "Yes",
        AXES[1]: "Partial",
        AXES[2]: "Yes",
        AXES[3]: "Yes",
        AXES[4]: "Partial",
        AXES[5]: "Partial",
        AXES[6]: "No",
        "core_positioning": "A custom workflow can reproduce pieces by gluing enrichment and heatmap tools together.",
        "scope_limitation": "Not a single maintained, citable, versioned package; reproducibility depends on bespoke glue code.",
    },
    {
        "tool": "HiMaLAYAS",
        AXES[0]: "Yes",
        AXES[1]: "Yes",
        AXES[2]: "Yes",
        AXES[3]: "Yes",
        AXES[4]: "Yes",
        AXES[5]: "Yes",
        AXES[6]: "Yes",
        "core_positioning": "Post hoc, multi-depth enrichment annotation of independently defined hierarchical matrix clusters.",
        "scope_limitation": "Uses ORA/hypergeometric enrichment; not a replacement for ranked-list GSEA or specialized network/module-discovery tools.",
    },
]

capability = pd.DataFrame(capability_rows).set_index("tool").loc[TOOLS].reset_index()
score_map_for_order = {"No": 0, "Not documented": 0, "Partial": 1, "Yes": 2}
capability["capability_score"] = (
    capability[AXES].replace(score_map_for_order).astype(int).sum(axis=1)
)
capability["tie_break_order"] = capability["tool"].map({tool: i for i, tool in enumerate(TOOLS)})
capability = (
    capability.sort_values(["capability_score", "tie_break_order"], ascending=[False, True])
    .drop(columns=["tie_break_order"])
    .reset_index(drop=True)
)
TOOLS = capability["tool"].tolist()
capability[["tool", "capability_score", "core_positioning", "scope_limitation"]]

## Render Supplementary Figure S4

A single capability heatmap is the figure.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.legend_handler import HandlerBase
from matplotlib.lines import Line2D
from matplotlib.patches import Circle, Wedge
from matplotlib.path import Path as MplPath

FONT = "Helvetica"
# Apply the shared publication font to every text element in this figure.
plt.rcParams["font.family"] = FONT
plt.rcParams["mathtext.fontset"] = "custom"
plt.rcParams["mathtext.rm"] = FONT
plt.rcParams["mathtext.it"] = f"{FONT}:italic"
plt.rcParams["mathtext.bf"] = f"{FONT}:bold"

TOOL_LABEL_FONTSIZE = 12
AXIS_LABEL_FONTSIZE = 12
DAGGER_FONTSIZE = 9
LEGEND_FONTSIZE = 11
LEGEND_MARKERSIZE = 9  # points; must match the Line2D markersize below
AXIS_LABEL_ROTATION = 35

PANEL_FIGSIZE = (13.0, 6.5)
BACKGROUND_COLOR = "white"
SCALE_COLORS = ["#ECEFF3", "#D9A441", "#2F6F73"]
SEPARATOR_COLOR = "#F5F5F5"
SEPARATOR_LW = 1.5
DOT_COLOR = "black"
DOT_EDGE_LW = 0.6
DOT_SIZE_PT2 = 100  # marker area in points^2, passed to scatter's `s`

score_map = {"No": 0, "Not documented": 0, "Partial": 1, "Yes": 2}
heat = capability.set_index("tool")[AXES].replace(score_map).astype(float)
himalayas_row = TOOLS.index("HiMaLAYAS")

fig, ax = plt.subplots(figsize=PANEL_FIGSIZE)
fig.patch.set_facecolor(BACKGROUND_COLOR)

cmap = ListedColormap(SCALE_COLORS)
norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5], cmap.N)
# interpolation="nearest" keeps cell edges crisp and pixel-aligned so the
# minor gridlines/HiMaLAYAS outline drawn on top land exactly on the color
# boundaries instead of appearing offset by antialiasing blur.
ax.imshow(heat.values, cmap=cmap, norm=norm, aspect="auto", interpolation="nearest")

ax.set_xticks(np.arange(len(AXES)))
ax.set_xticklabels(
    AXES, rotation=AXIS_LABEL_ROTATION, ha="right", va="top", fontsize=AXIS_LABEL_FONTSIZE
)
ax.set_yticks(np.arange(len(TOOLS)))
ax.set_yticklabels(TOOLS, fontsize=TOOL_LABEL_FONTSIZE)
ax.get_yticklabels()[himalayas_row].set_fontweight("bold")
ax.tick_params(length=0)

for spine in ax.spines.values():
    spine.set_visible(False)

# Subtle cell separators.
ax.set_xticks(np.arange(-0.5, len(AXES), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(TOOLS), 1), minor=True)
ax.grid(which="minor", color=SEPARATOR_COLOR, linewidth=SEPARATOR_LW, snap=True)
ax.tick_params(which="minor", bottom=False, left=False)

# Half-disc marker path (right half filled) for "Partial", built once.
_theta = np.linspace(np.pi / 2, -np.pi / 2, 30)
_half_verts = np.vstack([np.column_stack([np.cos(_theta), np.sin(_theta)]), [[0, -1], [0, 1]]])
HALF_DISC_MARKER = MplPath(_half_verts)

# One symbol system throughout, all in black: Yes -> filled dot, Partial ->
# half-filled dot, No / Not documented -> open dot. "Not documented"
# additionally gets a small dagger, since it is a distinct epistemic state
# (unverified, not checked-absent) worth keeping visible without
# reintroducing text-in-grid.
# Marker size is in points^2 (display space) so dots stay circular even
# though the heatmap axes are stretched to a non-square aspect.
yes_xy, partial_xy, null_xy, notdoc_xy = [], [], [], []
for y, tool in enumerate(TOOLS):
    for x, axis in enumerate(AXES):
        value = capability.loc[capability["tool"] == tool, axis].iloc[0]
        if value == "Yes":
            yes_xy.append((x, y))
        elif value == "Partial":
            partial_xy.append((x, y))
        else:
            null_xy.append((x, y))
            if value == "Not documented":
                notdoc_xy.append((x, y))

if yes_xy:
    yx, yy = zip(*yes_xy)
    ax.scatter(yx, yy, s=DOT_SIZE_PT2, marker="o", color=DOT_COLOR, zorder=3)

if partial_xy:
    px, py = zip(*partial_xy)
    ax.scatter(px, py, s=DOT_SIZE_PT2, marker=HALF_DISC_MARKER, color=DOT_COLOR, zorder=3)
    ax.scatter(
        px,
        tuple(map(lambda y: y + 0.01, py)),
        s=DOT_SIZE_PT2 + 15,
        marker="o",
        facecolors="none",
        edgecolors=DOT_COLOR,
        linewidths=DOT_EDGE_LW,
        zorder=3,
    )

if null_xy:
    nx, ny = zip(*null_xy)
    ax.scatter(
        nx,
        ny,
        s=DOT_SIZE_PT2,
        marker="o",
        facecolors="none",
        edgecolors=DOT_COLOR,
        linewidths=DOT_EDGE_LW,
        zorder=3,
    )

for x, y in notdoc_xy:
    ax.annotate(
        "†",
        (x, y),
        xytext=(6, 6),
        textcoords="offset points",
        fontsize=DAGGER_FONTSIZE,
        color=DOT_COLOR,
        ha="left",
        va="bottom",
        zorder=4,
    )


# Legend marker for "Partial" must match both the matrix's two-layer
# composition (filled half-disc + full outline circle) and the other
# legend markers' size. Line2D's markersize is a point-space diameter
# independent of the legend handle box, so the patch radius here is
# computed in points (via the figure's points->pixels scale) rather than
# from handle box height -- deriving it from height previously produced a
# visibly smaller circle whose wedge fill also poked past the outline.
class HalfCircleHandler(HandlerBase):
    def __init__(self, markersize_pt, edgewidth_pt, color):
        super().__init__()
        self.markersize_pt = markersize_pt
        self.edgewidth_pt = edgewidth_pt
        self.color = color

    def create_artists(
        self, legend, orig_handle, xdescent, ydescent, width, height, fontsize, trans
    ):
        cx = width / 2 - xdescent
        cy = height / 2 - ydescent
        px_per_pt = legend.figure.dpi / 72.0
        r = (self.markersize_pt / 2) * px_per_pt
        wedge = Wedge((cx, cy), r, 90, 270, facecolor=self.color, edgecolor="none", transform=trans)
        outline = Circle(
            (cx, cy),
            r,
            facecolor="none",
            edgecolor=self.color,
            linewidth=self.edgewidth_pt,
            transform=trans,
        )
        return [wedge, outline]


partial_proxy = Line2D([0], [0], linestyle="none", label="Partial")

legend_handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        linestyle="none",
        markerfacecolor=DOT_COLOR,
        markeredgecolor=DOT_COLOR,
        markersize=LEGEND_MARKERSIZE,
        label="Yes",
    ),
    partial_proxy,
    Line2D(
        [0],
        [0],
        marker="o",
        linestyle="none",
        markerfacecolor="none",
        markeredgecolor=DOT_COLOR,
        markeredgewidth=DOT_EDGE_LW,
        markersize=LEGEND_MARKERSIZE,
        label="No",
    ),
    Line2D(
        [0],
        [0],
        marker="o",
        linestyle="none",
        markerfacecolor="none",
        markeredgecolor=DOT_COLOR,
        markeredgewidth=DOT_EDGE_LW,
        markersize=LEGEND_MARKERSIZE,
        label="Not documented (†)",
    ),
]
ax.legend(
    handles=legend_handles,
    handler_map={
        partial_proxy: HalfCircleHandler(
            markersize_pt=LEGEND_MARKERSIZE - 2, edgewidth_pt=DOT_EDGE_LW, color=DOT_COLOR
        )
    },
    frameon=False,
    loc="upper left",
    bbox_to_anchor=(1.005, 1.0),
    fontsize=LEGEND_FONTSIZE,
    handletextpad=0.6,
    labelspacing=1.0,
)

fig.subplots_adjust(left=0.17, right=0.86, bottom=0.27, top=0.86)
output_path = PNG_DIR / "related_tool_capability_matrix.png"
fig.savefig(output_path, dpi=300, bbox_inches="tight", facecolor=BACKGROUND_COLOR)
print(f"Saved {output_path}")
plt.show()

## Sources

Primary sources for the capability entries above.

In [ ]:
sources = pd.DataFrame(
    [
        {
            "tool": "VisHiC",
            "source_type": "publication",
            "source": "Krushevskaya, Peterson, Reimand, Kull & Vilo (2009), Nucleic Acids Research 37:W587-W592; DOI 10.1093/nar/gkp435; PMID 19483095; PMCID PMC2703939.",
            "note": "Historical enrichment-guided microarray heatmap web tool; live-service status is dated.",
        },
        {
            "tool": "funcExplorer",
            "source_type": "publication/source archive",
            "source": "Kolberg, Kuzmin, Adler, Vilo & Peterson (2018), BMC Genomics 19(Suppl 1):817; DOI 10.1186/s12864-018-5176-x; PMID 30428831; PMCID PMC6236982. Source also archived at DOI 10.5281/zenodo.1194883.",
            "note": "Historical expression-data workflow; source recoverable, but not treated here as a simple local analysis package.",
        },
        {
            "tool": "WGCNA",
            "source_type": "publication",
            "source": "Langfelder & Horvath (2008), BMC Bioinformatics 9:559; DOI 10.1186/1471-2105-9-559; PMID 19114008; PMCID PMC2631488.",
            "note": "Mature module/network discovery package; enrichment and matrix-linked term rendering are separate downstream steps.",
        },
        {
            "tool": "clusterProfiler",
            "source_type": "publication/documentation",
            "source": "Wu et al. (2021), The Innovation 2(3):100141; DOI 10.1016/j.xinn.2021.100141; PMID 34557778; Bioconductor documentation.",
            "note": "Mature ORA/GSEA enrichment engine for supplied gene/ranked lists; no matrix clustering/rendering workflow.",
        },
        {
            "tool": "ComplexHeatmap",
            "source_type": "publication/documentation",
            "source": "Gu, Eils & Schlesner (2016), Bioinformatics 32(18):2847-2849; DOI 10.1093/bioinformatics/btw313; PMID 27207943; package documentation/README.",
            "note": "Mature matrix visualization package; can render supplied tracks but does not compute enrichment/FDR itself.",
        },
        {
            "tool": "Manual ORA-plus-heatmap",
            "source_type": "constructed workflow baseline",
            "source": "Not a single citable package; represents the common practice of gluing a standalone ORA/enrichment tool (e.g., clusterProfiler) to a standalone heatmap renderer (e.g., ComplexHeatmap) with custom looping code.",
            "note": "Baseline for what is achievable with existing mature tools and bespoke glue code, not a specific published implementation.",
        },
        {
            "tool": "HiMaLAYAS",
            "source_type": "source/API verification",
            "source": "Verified directly against the HiMaLAYAS source/API and this resubmission notebook package.",
            "note": "Capability claims concern package behavior and this yeast GI-PCC figure package.",
        },
    ]
)
sources

## Caption

**Supplementary Figure S4. Related-tool positioning.** Qualitative capability comparison between HiMaLAYAS and related workflows/tools. The figure is not a runtime, accuracy, biological-discovery, or usability benchmark. It summarizes whether each tool/workflow packages the specific capabilities needed for post hoc enrichment annotation of independently defined hierarchical matrix clusters: annotation after clustering, hierarchical zoom/multi-depth annotation, enrichment testing with multiple-testing control, matrix-linked significant-term rendering, domain generality beyond expression-specific workflows, and local reusable execution. The comparison supports a restrained novelty claim: HiMaLAYAS integrates these steps into one maintained workflow, while WGCNA, clusterProfiler, ComplexHeatmap, web tools, and manual ORA-plus-heatmap workflows cover important subsets or require custom glue code.

Symbols: a filled dot indicates **Yes** (documented/package-level capability); a half-filled dot indicates **Partial** (achievable only through custom looping/glue code, or supported for only part of the capability); an open dot indicates **No** (not part of the tool/workflow's documented role); an open dot marked with a dagger (†) indicates **Not documented** (not clearly established from the sources consulted, distinct from a confirmed absence). The orange outline marks the HiMaLAYAS row, the software being positioned by this comparison.